# DGA transformer fault diagnosis (Al-Sakini et al., Appl. Sci. 2025, 15, 118)
Upload DuVal.csv to Colab first (left sidebar > Files > Upload).

In [ ]:
# Cell 1 - imports and data
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix)

df = pd.read_csv("DuVal.csv")          # change path if needed
df["Fault"] = df["Fault"].str.strip()
print("Duplicates removed:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape); print(df["Fault"].value_counts())

In [ ]:
# Cell 2 - build the three input datasets (cases A, B, C)
g = df[["H2","CH4","C2H6","C2H4","C2H2","CO"]].astype(float)

# Case A - Duval triangle: % of CH4, C2H4, C2H2
s = g["CH4"] + g["C2H4"] + g["C2H2"]
A = pd.DataFrame({"pCH4": 100*g["CH4"]/s, "pC2H4": 100*g["C2H4"]/s, "pC2H2": 100*g["C2H2"]/s})

# Case B - Doernenburg: 4 ratios
B = pd.DataFrame({"C2H6/C2H2": g["C2H6"]/g["C2H2"], "C2H2/CH4": g["C2H2"]/g["CH4"],
                  "C2H2/C2H4": g["C2H2"]/g["C2H4"], "CH4/H2": g["CH4"]/g["H2"]})

# Case C - Rogers: 3 ratios
C = pd.DataFrame({"C2H2/C2H4": g["C2H2"]/g["C2H4"], "CH4/H2": g["CH4"]/g["H2"],
                  "C2H4/C2H6": g["C2H4"]/g["C2H6"]})

cases = {"A (Duval)": A, "B (Doernenburg)": B, "C (Rogers)": C}
y = df["Fault"].values
for k, X in cases.items():
    assert np.isfinite(X.values).all(), k      # no div-by-zero (min gas value is 0.01)

In [ ]:
# Cell 3 - models with the Table 4 parameters
def models():
    return {
        "RF":  RandomForestClassifier(n_estimators=25, max_depth=3, min_samples_split=10,
                                      max_features=None, random_state=42),
        "BPNN": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(100,),
                              activation="logistic", alpha=0.005, solver="sgd",
                              max_iter=1000, random_state=42)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(5, metric="euclidean", weights="uniform")),
        "SVM": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1, tol=1e-3)),
        "DT":  DecisionTreeClassifier(min_samples_leaf=2, max_depth=100, min_samples_split=5, random_state=42),
        "NB":  make_pipeline(StandardScaler(), GaussianNB(var_smoothing=2e-8)),
    }

In [ ]:
# Cell 4 - metrics (weighted averages, as Orange's Test & Score reports)
def specificity_weighted(yt, yp):
    labs = np.unique(yt); cm = confusion_matrix(yt, yp, labels=labs); tot = cm.sum(); out = []
    for i in range(len(labs)):
        tp = cm[i,i]; fp = cm[:,i].sum()-tp; fn = cm[i,:].sum()-tp; tn = tot-tp-fp-fn
        out.append(tn/(tn+fp) if tn+fp else 0)
    return np.average(out, weights=cm.sum(axis=1))

def metrics(yt, yp):
    return {"Accuracy": accuracy_score(yt, yp),
            "Precision": precision_score(yt, yp, average="weighted", zero_division=0),
            "Recall": recall_score(yt, yp, average="weighted", zero_division=0),
            "Specificity": specificity_weighted(yt, yp),
            "F1": f1_score(yt, yp, average="weighted", zero_division=0),
            "MCC": matthews_corrcoef(yt, yp)}

In [ ]:
# Cell 5 - run: 80/20 split, 10-fold CV on train, then test
results, cms = [], {}
for case, X in cases.items():
    Xtr, Xte, ytr, yte = train_test_split(X.values, y, test_size=0.2, random_state=42, stratify=y)
    skf = StratifiedKFold(10, shuffle=True, random_state=42)
    for name, m in models().items():
        # training stage (10-fold CV predictions on the 80%)
        ycv = cross_val_predict(m, Xtr, ytr, cv=skf)
        r = {"Case": case, "Method": name, "Stage": "Train(CV10)", **metrics(ytr, ycv)}
        results.append(r)
        # testing stage (fit on all 80%, predict the held-out 20%)
        m.fit(Xtr, ytr); yp = m.predict(Xte)
        results.append({"Case": case, "Method": name, "Stage": "Test", **metrics(yte, yp)})
        cms[(case, name)] = (confusion_matrix(yte, yp, labels=sorted(set(y))), sorted(set(y)))

res = pd.DataFrame(results)
pd.set_option("display.width", 200)
for case in cases:
    for stage in ["Train(CV10)", "Test"]:
        t = res[(res.Case==case)&(res.Stage==stage)].drop(columns=["Case","Stage"]).set_index("Method")
        print(f"\n=== Case {case} - {stage} ==="); print((t*100).round(1))

In [ ]:
# Cell 6 - confusion matrices + summary figure
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
fig, axs = plt.subplots(3, 6, figsize=(24, 12))
for i, case in enumerate(cases):
    for j, name in enumerate(["RF","BPNN","KNN","SVM","DT","NB"]):
        cm, labs = cms[(case, name)]
        ConfusionMatrixDisplay(cm, display_labels=labs).plot(ax=axs[i,j], colorbar=False, cmap="Blues")
        axs[i,j].set_title(f"{case} - {name}", fontsize=9)
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=120); plt.show()

test = res[res.Stage=="Test"].pivot(index="Method", columns="Case", values="Accuracy")*100
test.plot(kind="bar", figsize=(9,5), ylabel="Test accuracy (%)"); plt.tight_layout()
plt.savefig("test_accuracy.png", dpi=120); plt.show()
res.to_csv("all_results.csv", index=False)   # download from Files panel